In [ ]:
# House Prices - Advanced Regression Techniques
# Kaggle Competition Analysis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import warnings
import os
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("House Prices - Advanced Regression Techniques")
print("=" * 50)

# ============================================================================
# 1. DATA LOADING AND INITIAL EXPLORATION
# ============================================================================

def load_and_explore_data():
    """Load and perform initial exploration of the dataset"""
    print("\n1. LOADING AND EXPLORING DATA")
    print("-" * 30)
    
    # Load data
    try:
        train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
        test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
        print(f"✓ Training data loaded: {train_df.shape}")
        print(f"✓ Test data loaded: {test_df.shape}")
    except FileNotFoundError:
        print("Error: Please ensure 'train.csv' and 'test.csv' are in the working directory")
        return None, None
    
    # Basic information
    print(f"\nTraining set shape: {train_df.shape}")
    print(f"Test set shape: {test_df.shape}")
    print(f"Features: {train_df.shape[1] - 1}")
    
    # Target variable statistics
    print(f"\nSalePrice Statistics:")
    print(f"Mean: ${train_df['SalePrice'].mean():,.2f}")
    print(f"Median: ${train_df['SalePrice'].median():,.2f}")
    print(f"Std: ${train_df['SalePrice'].std():,.2f}")
    print(f"Min: ${train_df['SalePrice'].min():,.2f}")
    print(f"Max: ${train_df['SalePrice'].max():,.2f}")
    
    return train_df, test_df

# ============================================================================
# 2. DATA VISUALIZATION
# ============================================================================

def create_visualizations(train_df):
    """Create the required visualizations"""
    print("\n2. CREATING VISUALIZATIONS")
    print("-" * 30)
    
    # Set up the plotting area
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Histogram of SalePrice
    plt.subplot(2, 3, 1)
    plt.hist(train_df['SalePrice'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    plt.title('Distribution of SalePrice', fontsize=14, fontweight='bold')
    plt.xlabel('Sale Price ($)')
    plt.ylabel('Frequency')
    plt.ticklabel_format(style='plain', axis='x')
    
    # Add statistics text
    plt.axvline(train_df['SalePrice'].mean(), color='red', linestyle='--', 
                label=f'Mean: ${train_df["SalePrice"].mean():,.0f}')
    plt.axvline(train_df['SalePrice'].median(), color='green', linestyle='--', 
                label=f'Median: ${train_df["SalePrice"].median():,.0f}')
    plt.legend()
    
    # 2. Log-transformed SalePrice
    plt.subplot(2, 3, 2)
    plt.hist(np.log1p(train_df['SalePrice']), bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
    plt.title('Log-Transformed SalePrice Distribution', fontsize=14, fontweight='bold')
    plt.xlabel('Log(Sale Price + 1)')
    plt.ylabel('Frequency')
    
    # 3. Top 10 numerical features correlation with SalePrice
    numerical_features = train_df.select_dtypes(include=[np.number]).columns
    correlations = train_df[numerical_features].corr()['SalePrice'].sort_values(ascending=False)
    top_10_corr = correlations[1:11]  # Exclude SalePrice itself
    
    plt.subplot(2, 3, 3)
    colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(top_10_corr)))
    bars = plt.barh(range(len(top_10_corr)), top_10_corr.values, color=colors)
    plt.yticks(range(len(top_10_corr)), top_10_corr.index)
    plt.title('Top 10 Features Correlated with SalePrice', fontsize=14, fontweight='bold')
    plt.xlabel('Correlation Coefficient')
    
    # Add value labels on bars
    for i, (bar, value) in enumerate(zip(bars, top_10_corr.values)):
        plt.text(value + 0.01 if value > 0 else value - 0.01, i, f'{value:.3f}', 
                va='center', ha='left' if value > 0 else 'right')
    
    # 4. Heatmap of top 10 numerical features
    plt.subplot(2, 3, 4)
    top_features = ['SalePrice'] + list(top_10_corr.index[:9])
    correlation_matrix = train_df[top_features].corr()
    
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": .8}, fmt='.2f')
    plt.title('Correlation Heatmap - Top Features', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    # 5. Neighborhood vs SalePrice boxplot
    plt.subplot(2, 3, 5)
    neighborhood_prices = train_df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False)
    top_neighborhoods = neighborhood_prices.head(10).index
    
    data_to_plot = [train_df[train_df['Neighborhood'] == neighborhood]['SalePrice'].values 
                    for neighborhood in top_neighborhoods]
    
    box_plot = plt.boxplot(data_to_plot, labels=top_neighborhoods, patch_artist=True)
    plt.title('Top 10 Neighborhoods vs SalePrice', fontsize=14, fontweight='bold')
    plt.xlabel('Neighborhood')
    plt.ylabel('Sale Price ($)')
    plt.xticks(rotation=45, ha='right')
    plt.ticklabel_format(style='plain', axis='y')
    
    # Color the boxes
    colors = plt.cm.Set3(np.linspace(0, 1, len(box_plot['boxes'])))
    for box, color in zip(box_plot['boxes'], colors):
        box.set_facecolor(color)
        box.set_alpha(0.7)
    
    # 6. Overall Quality vs SalePrice
    plt.subplot(2, 3, 6)
    quality_prices = train_df.groupby('OverallQual').agg({
        'SalePrice': ['mean', 'count']
    }).round(0)
    quality_prices.columns = ['Mean_Price', 'Count']
    
    bars = plt.bar(quality_prices.index, quality_prices['Mean_Price'], 
                   alpha=0.7, color='lightgreen', edgecolor='black')
    plt.title('Overall Quality vs Average SalePrice', fontsize=14, fontweight='bold')
    plt.xlabel('Overall Quality Rating')
    plt.ylabel('Average Sale Price ($)')
    plt.ticklabel_format(style='plain', axis='y')
    
    # Add count labels on bars
    for bar, count in zip(bars, quality_prices['Count']):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'n={int(count)}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    return top_10_corr, top_features

# ============================================================================
# 3. DATA CLEANING AND FEATURE ENGINEERING
# ============================================================================

def clean_and_engineer_features(train_df, test_df):
    """Clean data and engineer features"""
    print("\n3. DATA CLEANING AND FEATURE ENGINEERING")
    print("-" * 45)
    
    # Combine datasets for consistent preprocessing
    train_len = len(train_df)
    combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    
    print(f"Combined dataset shape: {combined_df.shape}")
    
    # Missing value analysis
    missing_data = combined_df.isnull().sum().sort_values(ascending=False)
    missing_percent = (missing_data / len(combined_df)) * 100
    missing_df = pd.DataFrame({'Missing_Count': missing_data, 'Percentage': missing_percent})
    missing_df = missing_df[missing_df['Missing_Count'] > 0]
    
    print(f"\nFeatures with missing values: {len(missing_df)}")
    print("\nTop 10 features with most missing values:")
    print(missing_df.head(10))
    
    # Handle missing values strategically
    print("\n🔧 Handling missing values...")
    
    # Features where NaN means "None" or "No feature"
    none_features = ['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                     'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                     'PoolQC', 'Fence', 'MiscFeature']
    
    for feature in none_features:
        if feature in combined_df.columns:
            combined_df[feature].fillna('None', inplace=True)
    
    # Numerical features - fill with 0 where NaN means absence
    zero_features = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
                     'GarageYrBlt', 'GarageArea', 'GarageCars', 'MasVnrArea']
    
    for feature in zero_features:
        if feature in combined_df.columns:
            combined_df[feature].fillna(0, inplace=True)
    
    # Fill other missing values with mode/median
    for column in combined_df.columns:
        if combined_df[column].isnull().sum() > 0:
            if combined_df[column].dtype == 'object':
                combined_df[column].fillna(combined_df[column].mode()[0], inplace=True)
            else:
                combined_df[column].fillna(combined_df[column].median(), inplace=True)
    
    print(f"✓ Missing values handled. Remaining: {combined_df.isnull().sum().sum()}")
    
    # Feature Engineering
    print("\n🛠️ Engineering new features...")
    
    # Total area features
    combined_df['TotalSF'] = combined_df['1stFlrSF'] + combined_df['2ndFlrSF'] + combined_df['TotalBsmtSF']
    combined_df['TotalBathrooms'] = combined_df['FullBath'] + combined_df['HalfBath'] * 0.5 + combined_df['BsmtFullBath'] + combined_df['BsmtHalfBath'] * 0.5
    combined_df['TotalPorchSF'] = combined_df['OpenPorchSF'] + combined_df['EnclosedPorch'] + combined_df['3SsnPorch'] + combined_df['ScreenPorch']
    
    # Age features
    combined_df['HouseAge'] = combined_df['YrSold'] - combined_df['YearBuilt']
    combined_df['RemodAge'] = combined_df['YrSold'] - combined_df['YearRemodAdd']
    combined_df['IsRemodeled'] = (combined_df['YearBuilt'] != combined_df['YearRemodAdd']).astype(int)
    
    # Quality features
    combined_df['OverallScore'] = combined_df['OverallQual'] * combined_df['OverallCond']
    combined_df['QualityIndex'] = combined_df['OverallQual'] + combined_df['OverallCond']
    
    # Categorical feature engineering
    combined_df['HighValueNeighborhood'] = combined_df['Neighborhood'].isin(['NoRidge', 'NridgHt', 'StoneBr']).astype(int)
    
    print("✓ New features created:")
    new_features = ['TotalSF', 'TotalBathrooms', 'TotalPorchSF', 'HouseAge', 'RemodAge', 
                   'IsRemodeled', 'OverallScore', 'QualityIndex', 'HighValueNeighborhood']
    for feature in new_features:
        print(f"  - {feature}")
    
    # Encode categorical variables
    print("\n🔢 Encoding categorical variables...")
    
    categorical_features = combined_df.select_dtypes(include=['object']).columns.tolist()
    print(f"Categorical features to encode: {len(categorical_features)}")
    
    # Use Label Encoding for ordinal features and One-Hot for nominal
    ordinal_features = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 
                       'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
    
    # Create quality mappings
    quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    
    for feature in ordinal_features:
        if feature in combined_df.columns:
            combined_df[feature] = combined_df[feature].map(quality_map)
            combined_df[feature].fillna(0, inplace=True)
    
    # One-hot encode remaining categorical features
    remaining_categorical = [col for col in categorical_features if col not in ordinal_features]
    combined_df = pd.get_dummies(combined_df, columns=remaining_categorical, drop_first=True)
    
    print(f"✓ Final dataset shape: {combined_df.shape}")
    
    # Split back into train and test
    train_processed = combined_df[:train_len].copy()
    test_processed = combined_df[train_len:].copy()
    
    # Remove SalePrice from test set
    test_processed = test_processed.drop('SalePrice', axis=1)
    
    return train_processed, test_processed

# ============================================================================
# 4. OUTLIER DETECTION AND REMOVAL
# ============================================================================

def detect_and_handle_outliers(train_df):
    """Detect and handle outliers in the training data"""
    print("\n4. OUTLIER DETECTION AND HANDLING")
    print("-" * 35)
    
    # Create a copy to avoid modifying original
    train_clean = train_df.copy()
    
    # Outlier detection using IQR method for SalePrice
    Q1 = train_clean['SalePrice'].quantile(0.25)
    Q3 = train_clean['SalePrice'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = train_clean[(train_clean['SalePrice'] < lower_bound) | (train_clean['SalePrice'] > upper_bound)]
    print(f"Outliers detected using IQR method: {len(outliers)}")
    
    # Also detect outliers using Z-score for key features
    numerical_cols = train_clean.select_dtypes(include=[np.number]).columns
    z_score_outliers = []
    
    for col in ['GrLivArea', 'TotalSF', 'SalePrice']:
        if col in numerical_cols:
            z_scores = np.abs(stats.zscore(train_clean[col]))
            outliers_col = train_clean[z_scores > 3]
            z_score_outliers.extend(outliers_col.index.tolist())
    
    z_score_outliers = list(set(z_score_outliers))
    print(f"Outliers detected using Z-score (>3): {len(z_score_outliers)}")
    
    # Remove extreme outliers (very conservative approach)
    extreme_outliers = train_clean[
        (train_clean['GrLivArea'] > 4000) & (train_clean['SalePrice'] < 300000)
    ].index
    
    print(f"Extreme outliers to remove: {len(extreme_outliers)}")
    
    if len(extreme_outliers) > 0:
        train_clean = train_clean.drop(extreme_outliers)
        print(f"✓ Removed {len(extreme_outliers)} extreme outliers")
        print(f"New training set shape: {train_clean.shape}")
    
    return train_clean

# ============================================================================
# 5. MODEL TRAINING AND EVALUATION
# ============================================================================

def train_and_evaluate_models(train_df, target_col='SalePrice'):
    """Train multiple models and compare their performance"""
    print("\n5. MODEL TRAINING AND EVALUATION")
    print("-" * 35)
    
    # Prepare features and target
    X = train_df.drop([target_col, 'Id'], axis=1, errors='ignore')
    y = np.log1p(train_df[target_col])  # Log transform target
    
    print(f"Feature matrix shape: {X.shape}")
    print(f"Target vector shape: {y.shape}")
    
    # Split the data
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Scale features for neural network
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # Initialize models
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge Regression': Ridge(alpha=10.0),
        'Lasso Regression': Lasso(alpha=0.001),
        'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=10),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
        'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, eval_metric='rmse'),
        'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
        'CatBoost': cb.CatBoostRegressor(iterations=100, random_state=42, verbose=False),
        'Neural Network': MLPRegressor(hidden_layer_sizes=(100, 50), random_state=42, max_iter=500)
    }
    
    # Train and evaluate models
    results = {}
    feature_importance = {}
    
    print("\nTraining models...")
    print("-" * 20)
    
    for name, model in models.items():
        print(f"Training {name}...")
        
        try:
            # Use scaled data for Neural Network
            if name == 'Neural Network':
                model.fit(X_train_scaled, y_train)
                y_pred = model.predict(X_val_scaled)
            else:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_val)
            
            # Calculate metrics
            rmse = np.sqrt(mean_squared_error(y_val, y_pred))
            r2 = r2_score(y_val, y_pred)
            
            # Cross-validation score
            if name == 'Neural Network':
                cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, 
                                          scoring='neg_root_mean_squared_error')
            else:
                cv_scores = cross_val_score(model, X_train, y_train, cv=5, 
                                          scoring='neg_root_mean_squared_error')
            
            cv_rmse = -cv_scores.mean()
            cv_std = cv_scores.std()
            
            results[name] = {
                'Validation_RMSE': rmse,
                'R2_Score': r2,
                'CV_RMSE_Mean': cv_rmse,
                'CV_RMSE_Std': cv_std
            }
            
            # Extract feature importance (where available)
            if hasattr(model, 'feature_importances_'):
                importance = model.feature_importances_
                feature_names = X.columns
                feature_importance[name] = dict(zip(feature_names, importance))
            elif hasattr(model, 'coef_'):
                importance = np.abs(model.coef_)
                feature_names = X.columns
                feature_importance[name] = dict(zip(feature_names, importance))
            
            print(f"  ✓ RMSE: {rmse:.4f}, R²: {r2:.4f}, CV RMSE: {cv_rmse:.4f}±{cv_std:.4f}")
            
        except Exception as e:
            print(f"  ✗ Error training {name}: {str(e)}")
            results[name] = {'Validation_RMSE': np.nan, 'R2_Score': np.nan, 
                           'CV_RMSE_Mean': np.nan, 'CV_RMSE_Std': np.nan}
    
    return results, feature_importance, models, scaler, (X_train, X_val, y_train, y_val)

# ============================================================================
# 6. RESULTS VISUALIZATION AND MODEL SELECTION
# ============================================================================

def visualize_results_and_select_best_model(results, feature_importance):
    """Visualize model results and select the best model"""
    print("\n6. MODEL RESULTS AND SELECTION")
    print("-" * 32)
    
    # Create results DataFrame
    results_df = pd.DataFrame(results).T
    results_df = results_df.sort_values('Validation_RMSE')
    
    print("\nModel Performance Comparison:")
    print("=" * 50)
    print(results_df.round(4))
    
    # Find best model
    best_model_name = results_df.index[0]
    print(f"\n🏆 Best Model: {best_model_name}")
    print(f"   Validation RMSE: {results_df.loc[best_model_name, 'Validation_RMSE']:.4f}")
    print(f"   R² Score: {results_df.loc[best_model_name, 'R2_Score']:.4f}")
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. RMSE Comparison
    ax1 = axes[0, 0]
    colors = plt.cm.viridis(np.linspace(0, 1, len(results_df)))
    bars = ax1.barh(results_df.index, results_df['Validation_RMSE'], color=colors, alpha=0.8)
    ax1.set_xlabel('RMSE')
    ax1.set_title('Model Performance Comparison (RMSE)', fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for bar, value in zip(bars, results_df['Validation_RMSE']):
        if not np.isnan(value):
            ax1.text(value + 0.001, bar.get_y() + bar.get_height()/2, 
                    f'{value:.4f}', va='center', ha='left', fontsize=9)
    
    # 2. R² Score Comparison
    ax2 = axes[0, 1]
    bars2 = ax2.barh(results_df.index, results_df['R2_Score'], color=colors, alpha=0.8)
    ax2.set_xlabel('R² Score')
    ax2.set_title('Model Performance Comparison (R²)', fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for bar, value in zip(bars2, results_df['R2_Score']):
        if not np.isnan(value):
            ax2.text(value + 0.01, bar.get_y() + bar.get_height()/2, 
                    f'{value:.3f}', va='center', ha='left', fontsize=9)
    
    # 3. Feature Importance (for best model)
    ax3 = axes[1, 0]
    if best_model_name in feature_importance:
        importance_dict = feature_importance[best_model_name]
        # Get top 10 most important features
        sorted_importance = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)[:10]
        features, importance_values = zip(*sorted_importance)
        
        colors_feat = plt.cm.plasma(np.linspace(0, 1, len(features)))
        bars3 = ax3.barh(features, importance_values, color=colors_feat, alpha=0.8)
        ax3.set_xlabel('Feature Importance')
        ax3.set_title(f'Top 10 Feature Importance - {best_model_name}', fontweight='bold')
        ax3.grid(axis='x', alpha=0.3)
    else:
        ax3.text(0.5, 0.5, f'Feature importance not available\nfor {best_model_name}', 
                ha='center', va='center', transform=ax3.transAxes, fontsize=12)
        ax3.set_title('Feature Importance', fontweight='bold')
    
    # 4. Cross-Validation Scores with Error Bars
    ax4 = axes[1, 1]
    valid_results = results_df.dropna()
    x_pos = np.arange(len(valid_results))
    
    bars4 = ax4.bar(x_pos, valid_results['CV_RMSE_Mean'], 
                   yerr=valid_results['CV_RMSE_Std'], 
                   capsize=5, alpha=0.8, color=colors[:len(valid_results)])
    ax4.set_xlabel('Models')
    ax4.set_ylabel('CV RMSE')
    ax4.set_title('Cross-Validation RMSE (with std dev)', fontweight='bold')
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(valid_results.index, rotation=45, ha='right')
    ax4.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return best_model_name, results_df

# ============================================================================
# 7. GENERATE PREDICTIONS AND SUBMISSION FILE
# ============================================================================

def generate_predictions(best_model_name, models, scaler, train_data, test_data, target_col='SalePrice'):
    """Generate predictions using the best model"""
    print(f"\n7. GENERATING PREDICTIONS WITH {best_model_name.upper()}")
    print("-" * 50)
    
    # Prepare training data
    X_train_full = train_data.drop([target_col, 'Id'], axis=1, errors='ignore')
    y_train_full = np.log1p(train_data[target_col])
    
    # Prepare test data
    X_test = test_data.drop('Id', axis=1, errors='ignore')
    
    # Ensure same columns
    common_cols = X_train_full.columns.intersection(X_test.columns)
    X_train_full = X_train_full[common_cols]
    X_test = X_test[common_cols]
    
    print(f"Training on {X_train_full.shape[0]} samples with {X_train_full.shape[1]} features")
    print(f"Predicting on {X_test.shape[0]} test samples")
    
    # Get the best model and retrain on full training data
    best_model = models[best_model_name]
    
    # Use scaled data for Neural Network
    if best_model_name == 'Neural Network':
        scaler_full = RobustScaler()
        X_train_scaled = scaler_full.fit_transform(X_train_full)
        X_test_scaled = scaler_full.transform(X_test)
        
        best_model.fit(X_train_scaled, y_train_full)
        predictions_log = best_model.predict(X_test_scaled)
    else:
        best_model.fit(X_train_full, y_train_full)
        predictions_log = best_model.predict(X_test)
    
    # Transform predictions back to original scale
    predictions = np.expm1(predictions_log)
    
    print(f"✓ Predictions generated")
    print(f"  Mean prediction: ${predictions.mean():,.2f}")
    print(f"  Min prediction: ${predictions.min():,.2f}")
    print(f"  Max prediction: ${predictions.max():,.2f}")
    
    # Create submission file
    submission = pd.DataFrame({
        'Id': test_data['Id'],
        'SalePrice': predictions
    })
    
    # Save submission file to correct Kaggle path
    output_path = '/kaggle/working/submission.csv' if '/kaggle/' in os.getcwd() else 'submission.csv'
    submission.to_csv(output_path, index=False)
    print(f"✓ Submission file saved as '{output_path}'")
    print(f"  Shape: {submission.shape}")
    
    return submission, predictions

# ============================================================================
# 8. MAIN EXECUTION PIPELINE
# ============================================================================

def print_strategy_recommendations():
    """Print strategy recommendations"""
    print("\n📋 HOUSE PRICES COMPETITION STRATEGY")
    print("=" * 40)
    print("1. 📊 Data Analysis: Explore correlations and distributions")
    print("2. 🧹 Data Cleaning: Handle missing values intelligently")
    print("3. ⚙️  Feature Engineering: Create meaningful new features")
    print("4. 🎯 Outlier Detection: Remove extreme outliers carefully")
    print("5. 🤖 Model Selection: Test multiple algorithms")
    print("6. 🔍 Model Evaluation: Use cross-validation for robust assessment")
    print("7. 📤 Generate Predictions: Create submission file")
    print("-" * 40)

def main():
    """Main execution pipeline"""
    print("🏠 HOUSE PRICES - ADVANCED REGRESSION TECHNIQUES")
    print("=" * 55)
    print("Kaggle Competition Analysis Pipeline")
    
    # Step 1: Load and explore data
    train_df, test_df = load_and_explore_data()
    if train_df is None:
        return
    
    # Step 2: Create visualizations
    top_correlations, top_features = create_visualizations(train_df)
    
    # Step 3: Clean and engineer features
    train_processed, test_processed = clean_and_engineer_features(train_df, test_df)
    
    # Step 4: Handle outliers
    train_clean = detect_and_handle_outliers(train_processed)
    
    # Step 5: Train and evaluate models
    results, feature_importance, models, scaler, split_data = train_and_evaluate_models(train_clean)
    
    # Step 6: Visualize results and select best model
    best_model_name, results_df = visualize_results_and_select_best_model(results, feature_importance)
    
    # Step 7: Generate predictions and submission
    submission, predictions = generate_predictions(
        best_model_name, models, scaler, train_clean, test_processed
    )
    
    # Final summary
    print(f"\n🎯 FINAL SUMMARY")
    print("=" * 20)
    print(f"✓ Best Model: {best_model_name}")
    print(f"✓ Validation RMSE: {results_df.loc[best_model_name, 'Validation_RMSE']:.4f}")
    print(f"✓ R² Score: {results_df.loc[best_model_name, 'R2_Score']:.4f}")
    print(f"✓ Predictions generated for {len(predictions)} test samples")
    output_path = '/kaggle/working/submission.csv' if '/kaggle/' in os.getcwd() else 'submission.csv'
    print(f"✓ Submission file ready: {output_path}")
    
    print(f"\n📋 KEY INSIGHTS:")
    print(f"• Dataset: {train_df.shape[0]} training samples, {len(top_features)} key features")
    print(f"• Top correlated feature: {top_correlations.index[0]} (r={top_correlations.iloc[0]:.3f})")
    print(f"• Feature engineering: Added 9 new derived features")
    print(f"• Models tested: {len(results)} different algorithms")
    print(f"• Best performance: {best_model_name} with RMSE {results_df.loc[best_model_name, 'Validation_RMSE']:.4f}")
    
    return {
        'train_data': train_clean,
        'test_data': test_processed,
        'results': results_df,
        'best_model': best_model_name,
        'submission': submission,
        'feature_importance': feature_importance
    }

# ============================================================================
# 9. ADDITIONAL ANALYSIS FUNCTIONS
# ============================================================================

def advanced_feature_analysis(train_df, top_features):
    """Perform advanced feature analysis"""
    print("\n9. ADVANCED FEATURE ANALYSIS")
    print("-" * 32)
    
    # Feature distribution analysis
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.ravel()
    
    numerical_features = ['GrLivArea', 'TotalBsmtSF', 'GarageArea', 'YearBuilt', 'OverallQual']
    
    for i, feature in enumerate(numerical_features):
        if i < 6 and feature in train_df.columns:
            ax = axes[i]
            
            # Scatter plot with SalePrice
            scatter = ax.scatter(train_df[feature], train_df['SalePrice'], 
                               alpha=0.6, c=train_df['SalePrice'], 
                               cmap='viridis', s=20)
            
            # Add trend line
            z = np.polyfit(train_df[feature], train_df['SalePrice'], 1)
            p = np.poly1d(z)
            ax.plot(train_df[feature], p(train_df[feature]), "r--", alpha=0.8, linewidth=2)
            
            ax.set_xlabel(feature)
            ax.set_ylabel('SalePrice')
            ax.set_title(f'{feature} vs SalePrice')
            ax.grid(True, alpha=0.3)
            
            # Calculate and display correlation
            corr = train_df[feature].corr(train_df['SalePrice'])
            ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes, 
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Remove empty subplots
    for i in range(len(numerical_features), 6):
        axes[i].remove()
    
    plt.tight_layout()
    plt.show()

def create_model_comparison_table(results_df):
    """Create a formatted model comparison table"""
    print("\n📊 MODEL COMPARISON TABLE")
    print("=" * 80)
    
    # Create a formatted table
    table_data = []
    headers = ['Model', 'Validation RMSE', 'R² Score', 'CV RMSE Mean', 'CV RMSE Std', 'Rank']
    
    for i, (model_name, row) in enumerate(results_df.iterrows(), 1):
        table_data.append([
            model_name,
            f"{row['Validation_RMSE']:.4f}",
            f"{row['R2_Score']:.4f}",
            f"{row['CV_RMSE_Mean']:.4f}",
            f"{row['CV_RMSE_Std']:.4f}",
            f"#{i}"
        ])
    
    # Print formatted table
    col_widths = [max(len(str(row[i])) for row in [headers] + table_data) + 2 
                  for i in range(len(headers))]
    
    # Print header
    header_row = "|".join(h.center(col_widths[i]) for i, h in enumerate(headers))
    separator = "+".join("-" * col_widths[i] for i in range(len(headers)))
    
    print("+" + separator + "+")
    print("|" + header_row + "|")
    print("+" + separator + "+")
    
    # Print data rows
    for row in table_data:
        data_row = "|".join(str(row[i]).center(col_widths[i]) for i in range(len(row)))
        print("|" + data_row + "|")
    
    print("+" + separator + "+")

def create_submission_summary(submission, predictions):
    """Create a summary of the submission"""
    print(f"\n📤 SUBMISSION SUMMARY")
    print("=" * 25)
    
    output_path = '/kaggle/working/submission.csv' if '/kaggle/' in os.getcwd() else 'submission.csv'
    print(f"Submission file: {output_path}")
    print(f"Number of predictions: {len(predictions)}")
    print(f"Prediction statistics:")
    print(f"  Mean: ${predictions.mean():,.2f}")
    print(f"  Median: ${np.median(predictions):,.2f}")
    print(f"  Std: ${predictions.std():,.2f}")
    print(f"  Min: ${predictions.min():,.2f}")
    print(f"  Max: ${predictions.max():,.2f}")
    
    # Show first few predictions
    print(f"\nFirst 10 predictions:")
    for i in range(min(10, len(submission))):
        print(f"  ID {submission.iloc[i, 0]}: ${submission.iloc[i, 1]:,.2f}")

# ============================================================================
# RUN THE COMPLETE PIPELINE
# ============================================================================

if __name__ == "__main__":
    # Print strategy recommendations
    print_strategy_recommendations()
    
    # Check if data files exist
    if not (os.path.exists('/kaggle/input/house-prices-advanced-regression-techniques/train.csv') and os.path.exists('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')):
        print("\n⚠️  DATA FILES NOT FOUND")
        print("Please ensure 'train.csv' and 'test.csv' are in your working directory")
        print("Download them from: https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data")
        print("\n📝 SAMPLE USAGE:")
        print("1. Download the dataset from Kaggle")
        print("2. Place train.csv and test.csv in the same folder as this script")
        print("3. Run this script: python house_prices_analysis.py")
        print("4. Check the generated submission.csv file")
        print("5. Upload submission.csv to Kaggle competition")
    else:
        # Run the complete pipeline
        results = main()
        
        # Additional analysis
        if results:
            print("\n🔍 PERFORMING ADDITIONAL ANALYSIS...")
            advanced_feature_analysis(results['train_data'], ['GrLivArea', 'TotalBsmtSF'])
            create_model_comparison_table(results['results'])
            create_submission_summary(results['submission'], 
                                    results['submission']['SalePrice'].values)
            
            print(f"\n✅ ANALYSIS COMPLETE!")
            output_path = '/kaggle/working/submission.csv' if '/kaggle/' in os.getcwd() else 'submission.csv'
            print(f"Your submission file '{output_path}' is ready for Kaggle!")
            print(f"Competition URL: https://www.kaggle.com/c/house-prices-advanced-regression-techniques")